In [2]:
import os
import glob
import math
import numpy as np
from PIL import Image
import folium
from folium.raster_layers import ImageOverlay


# ---------------------------------------------------------
# CONFIGURATION
# ---------------------------------------------------------

TRAFFIC_DIR = "../../data/traffic"  # folder containing traffic PNGs

LAT = 48.14553552490184
LNG = 17.114780288953938
MAP_WIDTH = 2304
MAP_HEIGHT = 2304
MAP_ZOOM = 14


# ---------------------------------------------------------
# 1. Load Traffic Images
# ---------------------------------------------------------


def load_traffic_images(folder):
    image_paths = sorted(glob.glob(os.path.join(folder, "*.png")))

    if not image_paths:
        raise ValueError("No traffic images found in folder.")

    images = []
    for path in image_paths:
        img = Image.open(path).convert("RGBA")
        images.append(np.array(img))

    print(f"Loaded {len(images)} traffic images.")
    return images


# ---------------------------------------------------------
# 2. Aggregate Images (Preserve Traffic Colors)
# ---------------------------------------------------------


def aggregate_traffic_images(images):
    stack = np.stack(images, axis=0)

    rgb = stack[:, :, :, :3]
    alpha = stack[:, :, :, 3]

    # Mask for non-transparent pixels
    mask = alpha > 0

    # Count how many valid pixels per location
    count = np.maximum(mask.sum(axis=0), 1)

    # Sum only valid pixels
    summed = (rgb * mask[..., None]).sum(axis=0)

    # Average RGB values
    avg_rgb = summed / count[..., None]

    # Alpha channel: pixel visible if at least one image had data
    avg_alpha = (mask.sum(axis=0) > 0).astype(np.uint8) * 255

    result = np.dstack([avg_rgb.astype(np.uint8), avg_alpha])

    return Image.fromarray(result, mode="RGBA")


# ---------------------------------------------------------
# 3. Compute Image Bounds (Geographic Alignment)
# ---------------------------------------------------------


def calculate_image_bounds(center_lat, center_lng, zoom, width, height):
    scale_factor = 104300 / (2**zoom)

    lat_span = (height * scale_factor) / 111320
    lng_span = (width * scale_factor) / (
        111320 * abs(math.cos(math.radians(center_lat)))
    )

    bounds = [
        [center_lat - lat_span / 2, center_lng - lng_span / 2],  # bottom-left
        [center_lat + lat_span / 2, center_lng + lng_span / 2],  # top-right
    ]

    return bounds


# ---------------------------------------------------------
# 4. Build Aggregated Traffic Map
# ---------------------------------------------------------


def build_traffic_heatmap():
    # Load images
    images = load_traffic_images(TRAFFIC_DIR)

    # Aggregate across time
    aggregated_image = aggregate_traffic_images(images)

    # Save aggregated result
    output_path = "cache/aggregated_traffic.png"
    aggregated_image.save(output_path)
    print(f"Aggregated image saved to {output_path}")

    # Compute geographic bounds
    bounds = calculate_image_bounds(LAT, LNG, MAP_ZOOM, MAP_WIDTH, MAP_HEIGHT)

    # Create Folium map
    m = folium.Map(location=[LAT, LNG], zoom_start=MAP_ZOOM)

    # Overlay traffic raster
    ImageOverlay(
        image=output_path,
        bounds=bounds,
        opacity=0.85,
        interactive=False,
        cross_origin=False,
    ).add_to(m)

    return m


# ---------------------------------------------------------
# 5. Run
# ---------------------------------------------------------

if __name__ == "__main__":
    traffic_map = build_traffic_heatmap()
    traffic_map.save("cache/traffic_heatmap.html")

Loaded 2 traffic images.
Aggregated image saved to cache/aggregated_traffic.png
